In [4]:
# ============================================================
# Spam Message Detector
# Uses Naive Bayes + Bag-of-Words to classify SMS as spam/ham
# ============================================================

import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB          # Fixed: was 'MultinomiaNB'
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix




In [6]:
# STEP 2: CREATE A SMALL BEGINNER DATASET
# ---------------------------------------------------------

# We are creating the dataset manually so it can run directly in Colab
# label = spam or ham
# message = the text message

data = {
    "label": [
        "ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam",
        "ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam"
    ],
    "message": [
        "Hey, are we meeting today?",
        "Congratulations! You won a free iPhone. Click now!",
        "Please send me the notes of today class",
        "Win cash prize now by clicking this link",
        "Can you call me after lunch?",
        "Limited offer! Buy now and get 50 percent discount",
        "I am on the way to college",
        "You have been selected for a free vacation",
        "Let's study together in the evening",
        "Claim your lottery reward today",
        "Can you help me with Python homework?",
        "Urgent! Your account has won a huge reward",
        "Where are you now?",
        "Free entry in contest, click here now",
        "I will reach home by 8 PM",
        "Get a free gift voucher instantly",
        "Did you complete the assignment?",
        "Exclusive deal only for you, act fast",
        "Let's go for tea after class",
        "You are a lucky winner, claim your prize"
    ]
}

# Convert dictionary into DataFrame
df = pd.DataFrame(data)

# Show first 5 rows
print("First 5 rows of dataset:")
print(df.head())

First 5 rows of dataset:
  label                                            message
0   ham                         Hey, are we meeting today?
1  spam  Congratulations! You won a free iPhone. Click ...
2   ham            Please send me the notes of today class
3  spam           Win cash prize now by clicking this link
4   ham                       Can you call me after lunch?


In [7]:
# Check data set info

print("\nDataSet Shape")
print(df.shape)

print("\nColumn names:")
print(df.columns)

print("\nDataset info:")
print(df.info())

print("\nMissing values")
print(df.isnull().sum())


DataSet Shape
(20, 2)

Column names:
Index(['label', 'message'], dtype='object')

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    20 non-null     object
 1   message  20 non-null     object
dtypes: object(2)
memory usage: 452.0+ bytes
None

Missing values
label      0
message    0
dtype: int64


In [12]:
# Step 4: Text Cleaning Function
def clean_text(text):
    text = text.lower()                              # lowercase everything
    text = re.sub(r'\d+', '', text)                  # remove all digits
    text = re.sub(r'[^a-z\s]', '', text)             # Fixed: was '[^a-zAZ\s]' — keep only lowercase letters & spaces
    text = re.sub(r'\s+', ' ', text).strip()         # Fixed: was '' (merges words) and '.stripe()' (typo)
    return text

df["clean_message"] = df["message"].apply(clean_text)

print("\nDataset after text cleaning")    # Fixed: typo 'afte' → 'after'
print(df[["message", "clean_message"]].head())


Dataset after text cleaning
                                             message  \
0                         Hey, are we meeting today?   
1  Congratulations! You won a free iPhone. Click ...   
2            Please send me the notes of today class   
3           Win cash prize now by clicking this link   
4                       Can you call me after lunch?   

                                     clean_message  
0                         hey are we meeting today  
1  congratulations you won a free iphone click now  
2          please send me the notes of today class  
3         win cash prize now by clicking this link  
4                      can you call me after lunch  


In [15]:
# Step 5: Label Encoding
label_encoder = LabelEncoder()                                   # Fixed: instantiate before use if not done earlier

df["label_encoded"] = label_encoder.fit_transform(df["label"])  # Fixed: was 'label_encoding' — renamed to match print below

print("\nOriginal and encoded labels:")
print(df[["label", "label_encoded"]].head())

print("\nLabel mapping:")                                         # Fixed: was '\Label' (invalid escape) → '\n'
for i, class_name in enumerate(label_encoder.classes_):
    print(f"{class_name} --> {i}")


Original and encoded labels:
  label  label_encoded
0   ham              0
1  spam              1
2   ham              0
3  spam              1
4   ham              0

Label mapping:
ham --> 0
spam --> 1


In [16]:
# STEP 6: Seperate features and target

X = df["clean_message"]
y = df["label_encoded"]

print("\nFeatures sample:")
print(X.head())

print("\nTarget sample")
print(y.head())


Features sample:
0                           hey are we meeting today
1    congratulations you won a free iphone click now
2            please send me the notes of today class
3           win cash prize now by clicking this link
4                        can you call me after lunch
Name: clean_message, dtype: object

Target sample
0    0
1    1
2    0
3    1
4    0
Name: label_encoded, dtype: int64


In [17]:
# STEP 7: Train-Test Spilt

X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2,random_state=42
)

print("\nTrain-Test Split Shapes")
print("X_train:",X_train.shape)
print("X_test:",X_train.shape)
print("y_train:",y_train.shape)
print("y_test:",y_test)


Train-Test Split Shapes
X_train: (16,)
X_test: (16,)
y_train: (16,)
y_test: 0     0
17    1
15    1
1     1
Name: label_encoded, dtype: int64


In [19]:
# step 8 : Convert text into numbers

vectorizer = CountVectorizer()

# fit_transform on training data — learns vocab AND transforms
X_train_vectorized = vectorizer.fit_transform(X_train)

# transform only on test data — uses vocab learned from training
X_test_vectorized = vectorizer.transform(X_test)

print("\nShape after vectorization")
print("X_train_vectorized:", X_train_vectorized.shape)
print("X_test_vectorized:", X_test_vectorized.shape)

print("\nSample Vocabulary Words:")
print(list(vectorizer.vocabulary_.keys())[:18])


Shape after vectorization
X_train_vectorized: (16, 74)
X_test_vectorized: (4, 74)

Sample Vocabulary Words:
['lets', 'study', 'together', 'in', 'the', 'evening', 'limited', 'offer', 'buy', 'now', 'and', 'get', 'percent', 'discount', 'urgent', 'your', 'account', 'has']


In [20]:
# Step 9: Train the Model

model = MultinomialNB()
model.fit(X_train_vectorized, y_train)

print("\nModel Trained Successfully")


Model Trained Successfully


In [21]:
#Step 10: MAke predictions

#predict on test data
y_pred = model.predict(X_test_vectorized)

print("\nPredicted Values")
print(y_pred)

print("\nActual Values:")
print(y_test.values)


Predicted Values
[0 0 1 1]

Actual Values:
[0 1 1 1]


In [23]:
# STep 11 :Evaluate the Model

accuracy = accuracy_score(y_test,y_pred)

print("Accuracy of Model is:")
print(accuracy)
print("\nClassification Report:")
print(classification_report(y_test,y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test,y_pred))

Accuracy of Model is:
0.75

Classification Report:
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       1.00      0.67      0.80         3

    accuracy                           0.75         4
   macro avg       0.75      0.83      0.73         4
weighted avg       0.88      0.75      0.77         4


Confusion Matrix
[[1 0]
 [1 2]]


In [25]:
# Step 12: Test with new custom messages

new_messages = [
    "Congratulations you have won a free gift",  # Fix 1: added missing comma
    "Can u send me the class Link",
    "Claim Your Cash prize now",
    "I am coming in 10 minutes",
    "Exclusive Offer only today buy now"
]

cleaned_new_messages = [clean_text(msg) for msg in new_messages]

new_messages_vectorized = vectorizer.transform(cleaned_new_messages)  # Fix 2: vectorizer

new_predictions = model.predict(new_messages_vectorized)

print("\nNew Messages Prediction")
for msg, pred in zip(new_messages, new_predictions):
    print(f"Message: {msg}")
    label_name = label_encoder.inverse_transform([pred])[0]  # Fix 3: pred, Fix 4: indented
    print(f"Predicted Label: {label_name}\n")



New Messages Prediction
Message: Congratulations you have won a free gift
Predicted Label: spam

Message: Can u send me the class Link
Predicted Label: ham

Message: Claim Your Cash prize now
Predicted Label: spam

Message: I am coming in 10 minutes
Predicted Label: ham

Message: Exclusive Offer only today buy now
Predicted Label: spam

